# Prophet: All Datasets Benchmark with Hyperparameter Tuning

Runs Python Prophet and Go Prophet on **all 7 example datasets** with per-dataset hyperparameter tuning, then compares total wall-clock time and forecast quality.

| Dataset | Rows | Frequency | Characteristics |
|---------|------|-----------|-----------------|
| Peyton Manning | 2,905 | Daily | Strong weekly + yearly seasonality, trend changes |
| Wikipedia R | 2,863 | Daily | Programming language interest cycles |
| R Outliers 1 | 2,863 | Daily | Same as R with injected outliers |
| R Outliers 2 | 2,863 | Daily | Same as R with different outliers |
| Air Passengers | 144 | Monthly | Classic multiplicative seasonality |
| Retail Sales | 293 | Monthly | Macro-economic trend + seasonality |
| Pedestrians COVID | 1,490 | Daily | COVID shock, regime change |
| Yosemite Temps | 18,721 | Hourly | Sub-daily, temperature oscillations |

In [ ]:
import time, json, os, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from prophet import Prophet

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (14, 5), "font.size": 11})

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
EXAMPLES_DIR = os.path.join(PROJECT_DIR, "examples")
STAN_BINARY = os.path.join(PROJECT_DIR, "python", "prophet", "stan_model", "prophet_model.bin")
GO_DIR = os.path.join(PROJECT_DIR, "go")

print(f"Project: {PROJECT_DIR}")
print(f"Stan:    {STAN_BINARY}")
print(f"Go dir:  {GO_DIR}")

## 1. Dataset Definitions with Tuned Hyperparameters

Each dataset gets tailored Prophet settings based on its characteristics.

In [ ]:
DATASETS = [
    {
        "name": "Peyton Manning",
        "file": "example_wp_log_peyton_manning.csv",
        "predict_periods": 365,
        "freq": "D",
        "prophet_kwargs": dict(
            changepoint_prior_scale=0.05,
            seasonality_prior_scale=10,
            seasonality_mode="additive",
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
        ),
        "go_params": {
            "changepoint_prior_scale": 0.05,
            "seasonality_prior_scale": 10,
            "seasonality_mode": "additive",
            "growth": "linear",
            "n_changepoints": 25,
        },
    },
    {
        "name": "Wikipedia R",
        "file": "example_wp_log_R.csv",
        "predict_periods": 365,
        "freq": "D",
        "prophet_kwargs": dict(
            changepoint_prior_scale=0.1,
            seasonality_prior_scale=10,
            seasonality_mode="additive",
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
        ),
        "go_params": {
            "changepoint_prior_scale": 0.1,
            "seasonality_prior_scale": 10,
            "seasonality_mode": "additive",
            "growth": "linear",
            "n_changepoints": 25,
        },
    },
    {
        "name": "R Outliers 1",
        "file": "example_wp_log_R_outliers1.csv",
        "predict_periods": 365,
        "freq": "D",
        "prophet_kwargs": dict(
            changepoint_prior_scale=0.01,  # tighter to resist outliers
            seasonality_prior_scale=5,
            seasonality_mode="additive",
        ),
        "go_params": {
            "changepoint_prior_scale": 0.01,
            "seasonality_prior_scale": 5,
            "seasonality_mode": "additive",
            "growth": "linear",
            "n_changepoints": 25,
        },
    },
    {
        "name": "R Outliers 2",
        "file": "example_wp_log_R_outliers2.csv",
        "predict_periods": 365,
        "freq": "D",
        "prophet_kwargs": dict(
            changepoint_prior_scale=0.01,
            seasonality_prior_scale=5,
            seasonality_mode="additive",
        ),
        "go_params": {
            "changepoint_prior_scale": 0.01,
            "seasonality_prior_scale": 5,
            "seasonality_mode": "additive",
            "growth": "linear",
            "n_changepoints": 25,
        },
    },
    {
        "name": "Air Passengers",
        "file": "example_air_passengers.csv",
        "predict_periods": 36,   # 3 years monthly
        "freq": "MS",
        "prophet_kwargs": dict(
            changepoint_prior_scale=0.05,
            seasonality_prior_scale=10,
            seasonality_mode="multiplicative",  # classic multiplicative
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
        ),
        "go_params": {
            "changepoint_prior_scale": 0.05,
            "seasonality_prior_scale": 10,
            "seasonality_mode": "multiplicative",
            "growth": "linear",
            "n_changepoints": 10,
        },
    },
    {
        "name": "Retail Sales",
        "file": "example_retail_sales.csv",
        "predict_periods": 24,   # 2 years monthly
        "freq": "MS",
        "prophet_kwargs": dict(
            changepoint_prior_scale=0.5,   # flexible trend for macro
            seasonality_prior_scale=15,
            seasonality_mode="multiplicative",
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
        ),
        "go_params": {
            "changepoint_prior_scale": 0.5,
            "seasonality_prior_scale": 15,
            "seasonality_mode": "multiplicative",
            "growth": "linear",
            "n_changepoints": 15,
        },
    },
    {
        "name": "Pedestrians COVID",
        "file": "example_pedestrians_covid.csv",
        "predict_periods": 90,
        "freq": "D",
        "prophet_kwargs": dict(
            changepoint_prior_scale=0.5,   # flexible for COVID regime change
            seasonality_prior_scale=10,
            seasonality_mode="multiplicative",
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            n_changepoints=30,
        ),
        "go_params": {
            "changepoint_prior_scale": 0.5,
            "seasonality_prior_scale": 10,
            "seasonality_mode": "multiplicative",
            "growth": "linear",
            "n_changepoints": 30,
        },
    },
    {
        "name": "Yosemite Temps",
        "file": "example_yosemite_temps.csv",
        "predict_periods": 168,  # 1 week hourly
        "freq": "h",
        "prophet_kwargs": dict(
            changepoint_prior_scale=0.01,
            seasonality_prior_scale=5,
            seasonality_mode="additive",
            yearly_seasonality=False,
            weekly_seasonality=False,
            daily_seasonality=True,
        ),
        "go_params": {
            "changepoint_prior_scale": 0.01,
            "seasonality_prior_scale": 5,
            "seasonality_mode": "additive",
            "growth": "linear",
            "n_changepoints": 25,
        },
    },
]

print(f"Configured {len(DATASETS)} datasets for benchmarking")

## 2. Run Python Prophet on All Datasets

In [ ]:
py_results = []

for ds_cfg in DATASETS:
    name = ds_cfg["name"]
    path = os.path.join(EXAMPLES_DIR, ds_cfg["file"])
    df = pd.read_csv(path, parse_dates=["ds"])
    # Drop NaN y values (Yosemite has some)
    df = df.dropna(subset=["y"]).reset_index(drop=True)

    print(f"\n{'='*60}")
    print(f"  {name} ({len(df)} rows)")
    print(f"  Params: {ds_cfg['prophet_kwargs']}")
    print(f"{'='*60}")

    # --- Fit ---
    m = Prophet(**ds_cfg["prophet_kwargs"])
    fit_start = time.perf_counter()
    m.fit(df)
    fit_time = time.perf_counter() - fit_start
    print(f"  Fit:     {fit_time:.3f}s")

    # --- Predict ---
    future = m.make_future_dataframe(periods=ds_cfg["predict_periods"], freq=ds_cfg["freq"])
    pred_start = time.perf_counter()
    forecast = m.predict(future)
    pred_time = time.perf_counter() - pred_start
    print(f"  Predict: {pred_time:.3f}s ({len(future)} rows)")

    # --- Export for Go ---
    records = [{"ds": row["ds"].timestamp(), "y": float(row["y"])} for _, row in df.iterrows()]
    export_path = f"/tmp/prophet_bench_{ds_cfg['file'].replace('.csv', '.json')}"
    with open(export_path, "w") as f:
        json.dump(records, f)

    py_results.append({
        "name": name,
        "n_rows": len(df),
        "fit_time": fit_time,
        "predict_time": pred_time,
        "predict_rows": len(future),
        "export_path": export_path,
        "forecast": forecast,
        "df": df,
    })

py_total_fit = sum(r["fit_time"] for r in py_results)
py_total_pred = sum(r["predict_time"] for r in py_results)
print(f"\n{'='*60}")
print(f"  PYTHON TOTALS: fit={py_total_fit:.3f}s, predict={py_total_pred:.3f}s")
print(f"{'='*60}")

## 3. Run Go Prophet on All Datasets

In [ ]:
go_results = []
env = os.environ.copy()
env["PROPHET_STAN_BINARY"] = STAN_BINARY

for ds_cfg, py_r in zip(DATASETS, py_results):
    req = {
        "data_path": py_r["export_path"],
        "name": ds_cfg["name"],
        "predict_periods": ds_cfg["predict_periods"],
        **ds_cfg["go_params"],
    }

    result = subprocess.run(
        ["go", "run", "./cmd/bench_all", json.dumps(req)],
        cwd=GO_DIR, env=env, capture_output=True, text=True, timeout=300,
    )

    if result.returncode != 0:
        print(f"  FAILED {ds_cfg['name']}: {result.stderr.strip()}")
        go_results.append({"name": ds_cfg["name"], "fit_time_s": 0, "predict_time_s": 0})
        continue

    go_r = json.loads(result.stdout.strip())
    go_results.append(go_r)
    print(f"  {go_r['name']:20s}  fit={go_r['fit_time_s']:.3f}s  predict={go_r['predict_time_s']:.4f}s  ({go_r['n_days']} rows)")

go_total_fit = sum(r.get("fit_time_s", 0) for r in go_results)
go_total_pred = sum(r.get("predict_time_s", 0) for r in go_results)
print(f"\n{'='*60}")
print(f"  GO TOTALS: fit={go_total_fit:.3f}s, predict={go_total_pred:.4f}s")
print(f"{'='*60}")

## 4. Per-Dataset Speed Comparison

In [ ]:
# Build comparison dataframe
rows = []
for py_r, go_r in zip(py_results, go_results):
    go_fit = go_r.get("fit_time_s", 0)
    go_pred = go_r.get("predict_time_s", 0)
    rows.append({
        "Dataset": py_r["name"],
        "Rows": py_r["n_rows"],
        "Py Fit (s)": py_r["fit_time"],
        "Go Fit (s)": go_fit,
        "Fit Speedup": py_r["fit_time"] / go_fit if go_fit > 0 else 0,
        "Py Predict (s)": py_r["predict_time"],
        "Go Predict (s)": go_pred,
        "Predict Speedup": py_r["predict_time"] / go_pred if go_pred > 0 else 0,
    })

comp_df = pd.DataFrame(rows).set_index("Dataset")

# Add totals row
totals = pd.DataFrame([{
    "Dataset": "TOTAL",
    "Rows": comp_df["Rows"].sum(),
    "Py Fit (s)": comp_df["Py Fit (s)"].sum(),
    "Go Fit (s)": comp_df["Go Fit (s)"].sum(),
    "Fit Speedup": comp_df["Py Fit (s)"].sum() / comp_df["Go Fit (s)"].sum(),
    "Py Predict (s)": comp_df["Py Predict (s)"].sum(),
    "Go Predict (s)": comp_df["Go Predict (s)"].sum(),
    "Predict Speedup": comp_df["Py Predict (s)"].sum() / comp_df["Go Predict (s)"].sum(),
}]).set_index("Dataset")

comp_df = pd.concat([comp_df, totals])

comp_df.style.format({
    "Rows": "{:.0f}",
    "Py Fit (s)": "{:.3f}",
    "Go Fit (s)": "{:.3f}",
    "Fit Speedup": "{:.1f}x",
    "Py Predict (s)": "{:.3f}",
    "Go Predict (s)": "{:.4f}",
    "Predict Speedup": "{:.0f}x",
}).bar(subset=["Fit Speedup", "Predict Speedup"], color="#10b981", vmin=0)

## 5. Total Time: Python vs Go (All Datasets Combined)

In [ ]:
py_total = py_total_fit + py_total_pred
go_total = go_total_fit + go_total_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors_list = ["#3b82f6", "#10b981"]
labels = ["Python", "Go"]

# Fit total
ax = axes[0]
vals = [py_total_fit, go_total_fit]
bars = ax.bar(labels, vals, color=colors_list, width=0.45, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(vals) * 0.02,
            f"{val:.2f}s", ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_title(f"Total Fit Time\n{py_total_fit/go_total_fit:.1f}x faster", fontsize=13, fontweight="bold")
ax.set_ylabel("Seconds")

# Predict total
ax = axes[1]
vals = [py_total_pred, go_total_pred]
bars = ax.bar(labels, vals, color=colors_list, width=0.45, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, vals):
    lbl = f"{val:.3f}s" if val >= 0.01 else f"{val*1000:.1f}ms"
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(vals) * 0.02,
            lbl, ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_title(f"Total Predict Time\n{py_total_pred/go_total_pred:.0f}x faster", fontsize=13, fontweight="bold")
ax.set_ylabel("Seconds")

# Combined total
ax = axes[2]
vals = [py_total, go_total]
bars = ax.bar(labels, vals, color=colors_list, width=0.45, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(vals) * 0.02,
            f"{val:.2f}s", ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_title(f"Total (Fit + Predict)\n{py_total/go_total:.1f}x faster", fontsize=13, fontweight="bold")
ax.set_ylabel("Seconds")

fig.suptitle(f"Aggregate Benchmark — {len(DATASETS)} Datasets, {sum(r['n_rows'] for r in py_results):,} Total Rows",
             fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

## 6. Per-Dataset Fit Speedup

In [ ]:
names = [r["name"] for r in py_results]
py_fits = [r["fit_time"] for r in py_results]
go_fits = [r.get("fit_time_s", 0) for r in go_results]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
bars1 = ax.bar(x - width / 2, py_fits, width, label="Python", color="#3b82f6", edgecolor="white")
bars2 = ax.bar(x + width / 2, go_fits, width, label="Go", color="#10b981", edgecolor="white")

# Add speedup annotations
for i, (py_v, go_v) in enumerate(zip(py_fits, go_fits)):
    if go_v > 0:
        speedup = py_v / go_v
        ax.annotate(f"{speedup:.1f}x", xy=(i, max(py_v, go_v)),
                    xytext=(0, 8), textcoords="offset points",
                    ha="center", fontsize=10, fontweight="bold", color="#059669")

ax.set_ylabel("Time (seconds)")
ax.set_title("Fit Time per Dataset", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=25, ha="right")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Forecasts Gallery (All Datasets)

In [ ]:
n_datasets = len(py_results)
fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for i, py_r in enumerate(py_results):
    ax = axes[i]
    df_plot = py_r["df"]
    fc = py_r["forecast"]

    ax.scatter(df_plot["ds"], df_plot["y"], s=0.5, alpha=0.3, color="#64748b", zorder=2)
    ax.plot(fc["ds"], fc["yhat"], color="#2563eb", linewidth=1, zorder=3)
    ax.fill_between(fc["ds"], fc["yhat_lower"], fc["yhat_upper"], alpha=0.12, color="#2563eb")

    last_train = df_plot["ds"].max()
    ax.axvline(last_train, color="#ef4444", linestyle="--", linewidth=0.8, alpha=0.6)

    go_r = go_results[i]
    go_fit = go_r.get("fit_time_s", 0)
    speedup = py_r["fit_time"] / go_fit if go_fit > 0 else 0
    ax.set_title(f"{py_r['name']} ({py_r['n_rows']} rows) — Go {speedup:.1f}x faster",
                 fontsize=11, fontweight="bold")
    ax.tick_params(labelsize=9)

# Hide last subplot if odd number
if n_datasets < len(axes):
    for j in range(n_datasets, len(axes)):
        axes[j].set_visible(False)

fig.suptitle("Python Prophet Forecasts — All Datasets", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 8. Hyperparameter Tuning Summary

In [ ]:
hp_rows = []
for ds_cfg in DATASETS:
    kw = ds_cfg["prophet_kwargs"]
    hp_rows.append({
        "Dataset": ds_cfg["name"],
        "Growth": kw.get("growth", "linear"),
        "Seasonality Mode": kw.get("seasonality_mode", "additive"),
        "CP Prior τ": kw.get("changepoint_prior_scale", 0.05),
        "Season Prior σ": kw.get("seasonality_prior_scale", 10),
        "N Changepoints": kw.get("n_changepoints", 25),
        "Yearly": kw.get("yearly_seasonality", "auto"),
        "Weekly": kw.get("weekly_seasonality", "auto"),
        "Daily": kw.get("daily_seasonality", "auto"),
        "Forecast Periods": ds_cfg["predict_periods"],
    })

hp_df = pd.DataFrame(hp_rows).set_index("Dataset")
hp_df.style.set_caption("Per-Dataset Hyperparameter Configuration")